<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/generateText/NLP_DialoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

In [49]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [51]:
device = torch.device("cuda")
device

device(type='cuda')

In [52]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small", torch_dtype=torch.float32)
tokenizer.pad_token = tokenizer.eos_token
tokenizer

GPT2Tokenizer(name_or_path='microsoft/DialoGPT-small', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [53]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small",torch_dtype=torch.float32)
model

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [54]:
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [55]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [56]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [57]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [58]:
def combineText(example):
    return {
        "text": "User: " + example['context'] +
            "Bot: " + example['response']
    }

In [59]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [60]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [61]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [62]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [63]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [64]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [65]:
trainSet = trainSet.map(combineText)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [66]:
testSet = testSet.map(combineText)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [67]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [68]:
testSet = testSet.map(encode, batched=True)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [69]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [70]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [71]:
print(set(trainSet[0]["labels"]))

{640, 257, 644, 6792, 780, 2576, 2453, 20630, 534, 1560, 25, 284, 286, 1312, 4642, 1576, 1833, 428, 1839, 307, 12982, 3397, 326, 198, 329, 201, 21834, 460, 466, 470, 345, 475, 351, 611, 484, 1254, 616, 5737, 1641, 2408, 1265, 373, 2933, 502, 760, 1276, 765, 766, 16895}


In [72]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    eval_strategy='epoch',
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=False,
    max_grad_norm = 1.0
)

In [73]:
trainSet = trainSet.select(range(1100))

In [74]:
testSet = testSet.select(range(670))

In [88]:
inputs = tokenizer.encode(
	input(">> User: ") + " Bot: " + tokenizer.eos_token, return_tensors='pt'
).to(model.device)

outputs = model.generate(input_ids=inputs, max_length=100, max_new_tokens=100, do_sample=True, temperature=0.9, top_p=0.9, num_return_sequences=5)

>> User:  I'm feeling low. My dog just died. She helped me get through soooo many low points in life. How will I cope?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [89]:
tokenizer.decode(outputs, skip_special_tokens=True)

[" I'm feeling low. My dog just died. She helped me get through soooo many low points in life. How will I cope? Bot: Bot: what happened to your dog ?",
 " I'm feeling low. My dog just died. She helped me get through soooo many low points in life. How will I cope? Bot: Bot: how do you feel when you hear about someone that has been passed on the road and now you think about the dogs you had and how you're in the same boat but you've been in a constant struggle with a dog for a couple years now .",
 " I'm feeling low. My dog just died. She helped me get through soooo many low points in life. How will I cope? Bot: Bot: i'm in the same boat as you and your dog and i am a 35 year old man and my dog is my best friend who i can't stand to be alone anymoreBot: i'm sorry for your loss and i am sorry for your loss as well but there is always time to ventBot: i'm sorry for your loss and your suffering and the end of your marriageBot: i really don't think you'll find someone who you love anymore",


In [77]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [78]:
trainer.evaluate(testSet)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'eval_loss': 6.2974653244018555,
 'eval_model_preparation_time': 0.0034,
 'eval_runtime': 8.1828,
 'eval_samples_per_second': 81.879,
 'eval_steps_per_second': 81.879}

In [79]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,4.507103,3.551750,0.003400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=550, training_loss=4.454387290261009, metrics={'train_runtime': 140.8052, 'train_samples_per_second': 7.812, 'train_steps_per_second': 3.906, 'total_flos': 52231186022400.0, 'train_loss': 4.454387290261009, 'epoch': 1.0})

In [80]:
trainer.save_model("mental-health-dialogpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [81]:
trainer.evaluate(testSet)

{'eval_loss': 3.551750421524048,
 'eval_model_preparation_time': 0.0034,
 'eval_runtime': 8.2333,
 'eval_samples_per_second': 81.377,
 'eval_steps_per_second': 81.377,
 'epoch': 1.0}

In [82]:
fineTunedModel = AutoModelForCausalLM.from_pretrained("/content/mental-health-dialogpt", torch_dtype=torch.float32)
fineTunedModel

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [83]:
fineTunedTokenizer = AutoTokenizer.from_pretrained("/content/mental-health-dialogpt", torch_dtype=torch.float32)
fineTunedTokenizer.pad_token = fineTunedTokenizer.eos_token
fineTunedTokenizer

GPT2Tokenizer(name_or_path='/content/mental-health-dialogpt', vocab_size=0, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [90]:
inputs = fineTunedTokenizer.encode(
	input(">> User: ") + " Bot: " + fineTunedTokenizer.eos_token, return_tensors='pt'
).to(fineTunedModel.device)

outputs = fineTunedModel.generate(input_ids=inputs, max_new_tokens=100, do_sample=True, temperature=0.9, top_p=0.9, num_return_sequences=5)

>> User:  I'm feeling low. My dog just died. She helped me get through soooo many low points in life. How will I cope?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [91]:
fineTunedTokenizer.decode(outputs, skip_special_tokens=True)

['', '', '', '', '']